# 소스코드_1팀(이터널리턴_상위권_스쿼드_메타_분석)

이터널 리턴 아시아 랭크 스쿼드 데이터를 기반으로 상위권 Top3 성과를 설명하는 팀 조합, 캐릭터, 듀오 시너지, 패치 흐름을 분석한 제출용 정리본이다.

이 노트북은 기존 작업 노트북을 단순 병합한 파일이 아니라, 최종 제출과 발표 흐름에 맞춰 핵심 분석 코드를 하나의 흐름으로 재구성한 버전이다.


## 0. 제출용 노트북 구성

- 데이터 출처: Eternal Return 공식 개발자 API
- 분석 범위: 아시아 서버, 스쿼드 랭크, 상위권 표본, 2026-02-15 ~ 2026-03-15
- 분석 단위: 개인 참가자 row가 아니라 3인 팀 단위로 재구성한 team composition
- 주요 산출물: 역할 조합 분석, 캐릭터 일별 성과, 듀오 시너지, 패치별 변화, Top3 예측 모델, 웹 대시보드 데이터
- 대시보드: 역할 조합 / 캐릭터 분석 / 듀오 시너지 / 패치 타임라인 탭으로 제공


In [ ]:
from pathlib import Path
import json
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")

pd.set_option("display.max_columns", 80)
pd.set_option("display.width", 140)
pd.set_option("display.float_format", "{:.3f}".format)

plt.rcParams["font.family"] = "Malgun Gothic"
plt.rcParams["axes.unicode_minus"] = False

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "output").exists():
    PROJECT_ROOT = Path(r"C:\Users\tw2ps\Desktop\sparta_python\태블로 개인 프로젝트")

RUN_DIR = PROJECT_ROOT / "output" / "api_snowball_collect" / "team3_top300_500_asia_30d"
MART_DIR = RUN_DIR / "mart"
ML_DIR = RUN_DIR / "ml"
META_DIR = RUN_DIR / "meta"
WEB_DATA_DIR = PROJECT_ROOT / "web" / "src" / "data"

print("PROJECT_ROOT:", PROJECT_ROOT)
print("RUN_DIR:", RUN_DIR)


In [ ]:
required_files = {
    "collection_config": META_DIR / "collection_config.json",
    "summary_latest": META_DIR / "summary_latest.json",
    "story_feed": MART_DIR / "tableau_main_story_feed_v4_0.csv",
    "character_day": MART_DIR / "character_day_mart.csv",
    "duo_synergy": MART_DIR / "duo_synergy_mart.csv",
    "patch_compare": MART_DIR / "balanced_version_compare_labeled.csv",
    "model_metrics": ML_DIR / "top3_baseline_metrics.csv",
    "role_adjustment": ML_DIR / "ml_role_adjustment_by_role.csv",
}

file_check = pd.DataFrame([
    {
        "name": name,
        "exists": path.exists(),
        "size_mb": round(path.stat().st_size / 1024 / 1024, 2) if path.exists() else np.nan,
        "path": str(path.relative_to(PROJECT_ROOT)) if path.exists() else str(path),
    }
    for name, path in required_files.items()
])
file_check


## 1. 데이터 수집 기준

Eternal Return 공식 API에서 랭크 유저와 최근 게임 정보를 수집한 뒤, 스쿼드 3인 팀만 필터링하였다.

수집 단계에서는 API 호출 제한과 중복 게임 수집을 고려해 frontier 방식으로 유저와 게임을 확장하였고, 이후 분석 단계에서는 동일한 팀 단위로 재구성된 mart 파일을 기준 데이터로 사용하였다.


In [ ]:
with open(required_files["collection_config"], encoding="utf-8") as f:
    collection_config = json.load(f)
with open(required_files["summary_latest"], encoding="utf-8") as f:
    summary_latest = json.load(f)

summary_table = pd.DataFrame([
    ["server", ", ".join(summary_latest.get("allowed_server_names", []))],
    ["matching_team_mode", summary_latest.get("matching_team_mode")],
    ["participant_rows_collected", summary_latest.get("participant_rows_collected")],
    ["total_games_seen", summary_latest.get("total_games_seen")],
    ["total_game_details_fetched", summary_latest.get("total_game_details_fetched")],
    ["participant_date_min", summary_latest.get("participant_date_min")],
    ["participant_date_max", summary_latest.get("participant_date_max")],
    ["api_calls_last_run", summary_latest.get("api_calls")],
], columns=["item", "value"])

summary_table


### API 수집 코드 구조

실제 API 키는 제출물에 포함하지 않고 환경변수로만 주입한다.

아래 코드는 수집 스크립트의 핵심 구조를 제출용으로 요약한 형태이며, 네트워크 호출은 API 키가 있을 때만 실행하도록 분리하였다.


In [ ]:
import os
import time
import requests

ER_API_BASE_URL = "https://open-api.bser.io/v1"


def build_er_headers() -> dict:
    api_key = os.getenv("ER_API_KEY")
    if not api_key:
        raise RuntimeError("ER_API_KEY 환경변수가 필요하다. 제출 노트북에는 API 키를 포함하지 않는다.")
    return {"x-api-key": api_key}


def er_api_get(endpoint: str, params: dict | None = None, sleep_seconds: float = 1.1) -> dict:
    url = f"{ER_API_BASE_URL}/{endpoint.lstrip('/')}"
    response = requests.get(url, headers=build_er_headers(), params=params, timeout=30)
    time.sleep(sleep_seconds)
    response.raise_for_status()
    return response.json()

collection_script = PROJECT_ROOT / "scripts" / "er_api_collect_snowball.py"
print("수집 스크립트 존재 여부:", collection_script.exists())
print("제출 노트북에서는 기존 수집 결과 mart를 기준으로 분석을 재현한다.")


In [ ]:
config_table = pd.DataFrame([
    ["top_limit", collection_config.get("top_limit")],
    ["rank_offset", collection_config.get("rank_offset")],
    ["min_start_date", collection_config.get("min_start_date")],
    ["max_start_date", collection_config.get("max_start_date")],
    ["allowed_server_names", ", ".join(collection_config.get("allowed_server_names", []))],
    ["max_retries", collection_config.get("max_retries")],
    ["min_interval", collection_config.get("min_interval")],
], columns=["config", "value"])

config_table


## 2. 전처리와 분석 mart 구성

원천 데이터는 참가자 단위 row이므로 팀 성과를 직접 설명하기 어렵다.

따라서 `gameId + teamNumber`를 기준으로 3인 팀을 구성하고, 캐릭터/무기/역할/전술 스킬/특성 정보를 팀 단위 signature로 재배열하였다.

최종 분석 mart는 대시보드에서 바로 읽을 수 있도록 역할 조합, 캐릭터 일별 성과, 듀오 시너지, 패치 비교 단위로 분리하였다.


In [ ]:
mart_files = [
    "team_comp_structure_mart.csv",
    "team_game_mart.csv",
    "team_role_structure_mart.csv",
    "team_character_weapon_role_mart.csv",
    "character_day_mart.csv",
    "duo_synergy_mart.csv",
    "balanced_version_compare_labeled.csv",
    "tableau_main_story_feed_v4_0.csv",
]

mart_audit = []
for file_name in mart_files:
    path = MART_DIR / file_name
    if path.exists():
        sample = pd.read_csv(path, nrows=5)
        mart_audit.append({
            "file": file_name,
            "size_mb": round(path.stat().st_size / 1024 / 1024, 2),
            "columns": sample.shape[1],
            "sample_rows_loaded": sample.shape[0],
        })
    else:
        mart_audit.append({"file": file_name, "size_mb": np.nan, "columns": np.nan, "sample_rows_loaded": 0})

pd.DataFrame(mart_audit)


### 코드북 및 역할 매핑

API 원천 데이터의 캐릭터/무기 코드는 그대로는 발표와 대시보드에서 해석하기 어렵다.

따라서 캐릭터 한글명, 대표 무기, 무기 기반 역할 태그를 매핑하여 `탱커`, `브루저`, `원거리 딜러`, `메이지` 같은 팀 구조 분석 단위로 변환하였다.


In [ ]:
weapon_mapping_path = WEB_DATA_DIR / "character_weapon_mapping_manual.csv"
weapon_mapping = pd.read_csv(weapon_mapping_path)

mapping_summary = (
    weapon_mapping
    .groupby(["role_main", "review_status"], dropna=False)
    .size()
    .reset_index(name="mapping_count")
    .sort_values(["mapping_count", "role_main"], ascending=[False, True])
)

mapping_summary.head(12)


In [ ]:
weapon_mapping[[
    "characterNum", "name_ko", "weapon_name_ko", "role_main", "role_alt",
    "mapping_source", "review_status", "pick_count", "game_count",
]].sort_values("pick_count", ascending=False).head(12)


## 3. EDA: 역할 조합 성과

역할 조합 탭은 캐릭터 티어가 아니라 팀 구조가 Top3 성과를 얼마나 설명하는지 확인하기 위한 화면이다.

`primary_top3_rate`와 `primary_team_count`를 함께 보며, 표본이 충분하면서 평균 대비 Top3 진입률이 높은 조합을 우선 해석하였다.


In [ ]:
story = pd.read_csv(required_files["story_feed"])
role_story = (
    story[story["section"].eq("역할 조합")]
    .sort_values(["primary_top3_rate", "primary_team_count"], ascending=[False, False])
    .reset_index(drop=True)
)

role_cols = [
    "primary_signature", "primary_team_count", "primary_team_share_pct",
    "primary_top3_rate", "primary_top3_rate_diff", "primary_win_rate", "primary_avg_rank",
    "secondary_signature", "secondary_top3_rate",
]
role_top = role_story[role_cols].head(10)
role_top


In [ ]:
plot_df = role_top.sort_values("primary_top3_rate", ascending=True)

fig, ax = plt.subplots(figsize=(10, 5))
ax.barh(plot_df["primary_signature"], plot_df["primary_top3_rate"], color="#44d7c6")
ax.axvline(story["primary_top3_rate"].mean(), color="#94a3b8", linestyle="--", linewidth=1)
ax.set_title("역할 조합 Top3 진입률 상위 구조")
ax.set_xlabel("Top3 rate (%)")
ax.set_ylabel("")
plt.tight_layout()
plt.show()


## 4. EDA: 캐릭터 일별 성과

캐릭터 분석은 단일 캐릭터의 절대 티어를 선언하는 방식이 아니라, 선택률과 Top3 성과가 날짜와 패치 흐름에 따라 어떻게 변하는지 보는 방식으로 구성하였다.

대시보드에서는 캐릭터 아이콘, 픽률, Top3 진입률, 승률, 평균 순위를 함께 보여주도록 가공하였다.


In [ ]:
character_day = pd.read_csv(required_files["character_day"])
latest_day = character_day["play_date"].max()
latest_character = character_day[character_day["play_date"].eq(latest_day)].copy()

latest_character_top = (
    latest_character[latest_character["team_count"].ge(30)]
    .sort_values(["top3_rate", "pick_count"], ascending=[False, False])
    [["play_date", "version", "character_name", "pick_count", "pick_share", "top3_rate", "win_rate", "avg_rank"]]
    .head(12)
)

latest_character_top


In [ ]:
plot_df = latest_character_top.sort_values("top3_rate", ascending=True)

fig, ax = plt.subplots(figsize=(10, 5))
ax.barh(plot_df["character_name"], plot_df["top3_rate"], color="#9bb5ff")
ax.set_title(f"최신 일자 캐릭터 Top3 진입률 상위권 ({latest_day})")
ax.set_xlabel("Top3 rate (%)")
ax.set_ylabel("")
plt.tight_layout()
plt.show()


## 5. EDA: 듀오 시너지

듀오 시너지는 3인 팀 내부에서 함께 등장한 두 캐릭터 조합의 성과를 측정한 지표이다.

단순 Top3 비율만 보면 표본이 작은 조합이 과대평가될 수 있으므로, `pair_count` 하한을 적용한 뒤 성과를 비교하였다.


In [ ]:
duo = pd.read_csv(required_files["duo_synergy"])
MIN_PAIR_COUNT = 40

duo_top = (
    duo[duo["pair_count"].ge(MIN_PAIR_COUNT)]
    .assign(pair=lambda x: x["character_a_name"] + " + " + x["character_b_name"])
    .sort_values(["top3_rate", "pair_count"], ascending=[False, False])
    [["play_date", "version", "pair", "pair_count", "pair_share", "top3_rate", "win_rate", "avg_rank"]]
    .head(12)
)

duo_top


In [ ]:
plot_df = duo_top.sort_values("top3_rate", ascending=True)

fig, ax = plt.subplots(figsize=(10, 5))
ax.barh(plot_df["pair"], plot_df["top3_rate"], color="#ffc27a")
ax.set_title(f"듀오 시너지 Top3 진입률 상위 조합 (pair_count >= {MIN_PAIR_COUNT})")
ax.set_xlabel("Top3 rate (%)")
ax.set_ylabel("")
plt.tight_layout()
plt.show()


## 6. 패치 버전별 변화 분석

패치 타임라인은 특정 캐릭터가 한 시점에서만 강한지, 여러 패치에 걸쳐 상승 또는 하락 흐름을 보이는지 확인하기 위한 보조 지표이다.

버전 2.0, 3.0, 4.0의 픽률과 Top3 성과를 비교해 `지속 상승`, `지속 하락`, `정체·하락` 같은 변화 유형을 부여하였다.


In [ ]:
patch_compare = pd.read_csv(required_files["patch_compare"])

patch_focus = (
    patch_compare
    .sort_values("delta_top3_4_vs_3", ascending=False)
    [[
        "character_name", "pick_share_2.0", "pick_share_3.0", "pick_share_4.0",
        "top3_rate_2.0", "top3_rate_3.0", "top3_rate_4.0",
        "delta_top3_4_vs_3", "meta_shift_refined", "patch_impact_type",
    ]]
    .head(12)
)

patch_focus


In [ ]:
plot_df = patch_focus.sort_values("delta_top3_4_vs_3", ascending=True)

fig, ax = plt.subplots(figsize=(10, 5))
colors = np.where(plot_df["delta_top3_4_vs_3"] >= 0, "#44d7c6", "#fb7185")
ax.barh(plot_df["character_name"], plot_df["delta_top3_4_vs_3"], color=colors)
ax.axvline(0, color="#94a3b8", linewidth=1)
ax.set_title("패치 4.0 기준 Top3 진입률 변화폭 (vs 3.0)")
ax.set_xlabel("Top3 rate change, pp")
ax.set_ylabel("")
plt.tight_layout()
plt.show()


## 7. 모델링: Top3 성과 예측 기준

머신러닝은 대시보드의 핵심 분석을 대체하기보다, 팀 구조 정보가 실제 Top3 성과를 어느 정도 설명하는지 검증하는 보조 장치로 사용하였다.

모델 입력에서는 `gameRank`, `victory`, `teamKill`, `monsterKill`, `avg_mmrGain`, `avg_mmrAfter`처럼 경기 결과 이후에 알 수 있는 누수 컬럼을 제외하였다.

최종 모델은 패치가 바뀌어도 설명력이 남는지 확인하기 위해 `2.0 + 3.0 학습 → 4.0 테스트` 기준을 중요하게 보았다.


In [ ]:
metrics = pd.read_csv(required_files["model_metrics"])
metric_view = (
    metrics
    .sort_values(["split", "roc_auc"], ascending=[True, False])
    [[
        "split", "model", "train_rows", "test_rows", "test_positive_rate",
        "accuracy", "precision", "recall", "f1", "roc_auc", "average_precision", "top_decile_lift",
    ]]
)

metric_view


In [ ]:
best_patch_model = (
    metrics[metrics["split"].eq("patch_2_3_to_4")]
    .sort_values("roc_auc", ascending=False)
    .head(1)
    .T
)

best_patch_model


### 선택 실행: LightGBM 재학습 코드

아래 셀은 제출 노트북에서 모델 구현 방식을 확인할 수 있도록 포함한 재학습 코드이다.

기본값은 `RUN_FULL_TRAINING = False`로 두어 노트북을 빠르게 열 수 있게 하였고, 필요 시 True로 바꾸면 `team_comp_structure_mart.csv`를 사용해 재학습한다.


In [ ]:
RUN_FULL_TRAINING = False

if RUN_FULL_TRAINING:
    from sklearn.compose import ColumnTransformer
    from sklearn.impute import SimpleImputer
    from sklearn.metrics import roc_auc_score, average_precision_score, classification_report
    from sklearn.model_selection import train_test_split
    from sklearn.pipeline import Pipeline
    from sklearn.preprocessing import OneHotEncoder, StandardScaler
    from lightgbm import LGBMClassifier

    team_comp = pd.read_csv(MART_DIR / "team_comp_structure_mart.csv")
    team_comp["version"] = team_comp["version"].astype(str)

    categorical_features = [
        "version",
        "weapon_role_signature_sorted",
        "weapon_signature_sorted",
        "character_weapon_signature_sorted",
        "tactical_1_name", "tactical_2_name", "tactical_3_name",
        "traitcore_1_name", "traitcore_2_name", "traitcore_3_name",
    ]
    numeric_features = [
        "avg_rankPoint", "avg_mmrBefore", "weapon_role_nunique",
        "frontline_cnt", "backline_cnt", "has_frontline", "has_backline",
        "is_double_frontline", "is_double_backline", "is_three_unique_roles",
    ]

    feature_cols = categorical_features + numeric_features
    train_mask = team_comp["version"].isin(["2.0", "3.0"])
    test_mask = team_comp["version"].eq("4.0")

    X_train = team_comp.loc[train_mask, feature_cols]
    y_train = team_comp.loc[train_mask, "is_top3"]
    X_test = team_comp.loc[test_mask, feature_cols]
    y_test = team_comp.loc[test_mask, "is_top3"]

    preprocessor = ColumnTransformer(
        transformers=[
            ("cat", Pipeline([
                ("imputer", SimpleImputer(strategy="most_frequent")),
                ("onehot", OneHotEncoder(handle_unknown="ignore", min_frequency=20)),
            ]), categorical_features),
            ("num", Pipeline([
                ("imputer", SimpleImputer(strategy="median")),
                ("scaler", StandardScaler()),
            ]), numeric_features),
        ]
    )

    model = Pipeline([
        ("prep", preprocessor),
        ("model", LGBMClassifier(
            n_estimators=700,
            learning_rate=0.025,
            num_leaves=31,
            min_child_samples=80,
            subsample=0.85,
            colsample_bytree=0.85,
            class_weight="balanced",
            random_state=42,
            n_jobs=-1,
        )),
    ])

    model.fit(X_train, y_train)
    pred_proba = model.predict_proba(X_test)[:, 1]
    pred_label = (pred_proba >= 0.5).astype(int)

    print("ROC-AUC:", roc_auc_score(y_test, pred_proba))
    print("Average Precision:", average_precision_score(y_test, pred_proba))
    print(classification_report(y_test, pred_label))
else:
    print("재학습은 기본 비활성화 상태이다. RUN_FULL_TRAINING 값을 True로 바꾸면 실행된다.")


## 8. 모델 결과의 대시보드 활용

모델 예측값은 단순히 “맞췄다”로 끝내지 않고, 실제 성과와 기대 성과의 차이를 비교하는 방식으로 활용하였다.

예를 들어 어떤 역할 조합의 실제 Top3 진입률이 모델 기대값보다 높으면, 해당 조합은 표본 조건을 감안해도 성과가 좋은 구조로 해석할 수 있다.


In [ ]:
role_adjustment = pd.read_csv(required_files["role_adjustment"])
role_adjustment_top = (
    role_adjustment[role_adjustment["team_count"].ge(100)]
    .sort_values("adjusted_diff_pp", ascending=False)
    .head(12)
)

role_adjustment_top


In [ ]:
plot_df = role_adjustment_top.sort_values("adjusted_diff_pp", ascending=True)

fig, ax = plt.subplots(figsize=(10, 5))
ax.barh(plot_df["role_signature"], plot_df["adjusted_diff_pp"], color="#44d7c6")
ax.axvline(0, color="#94a3b8", linewidth=1)
ax.set_title("모델 기대값 대비 실제 Top3 성과가 높은 역할 조합")
ax.set_xlabel("Actual - Expected Top3 rate, pp")
ax.set_ylabel("")
plt.tight_layout()
plt.show()


## 9. 웹 대시보드 산출 데이터

분석 결과는 최종적으로 웹 대시보드가 바로 읽을 수 있는 csv/json 형태로 정리하였다.

대시보드는 역할 조합, 캐릭터 분석, 듀오 시너지, 패치 타임라인을 한 화면 흐름 안에서 탐색할 수 있도록 구성하였다.


In [ ]:
web_files = [
    "tableau_main_story_feed_v4_0.csv",
    "character_day_mart.csv",
    "duo_synergy_mart.csv",
    "balanced_version_compare_labeled.csv",
    "weapon_role_story_shortlist_drilldown_v4_0_top5.csv",
    "role-duo-validation.json",
    "ml-role-adjustment.json",
    "ai-role-briefings.json",
    "timeline-meta-briefing.json",
    "character-icons.json",
]

web_audit = pd.DataFrame([
    {
        "dashboard_file": file_name,
        "exists": (WEB_DATA_DIR / file_name).exists(),
        "size_mb": round((WEB_DATA_DIR / file_name).stat().st_size / 1024 / 1024, 2) if (WEB_DATA_DIR / file_name).exists() else np.nan,
    }
    for file_name in web_files
])

web_audit


In [ ]:
print("대시보드 주요 해석 축")
print("1. 역할 조합: 팀 구조 단위 Top3 성과와 무기 보조 구조 확인")
print("2. 캐릭터 분석: 캐릭터별 픽률, Top3 진입률, 패치별 변화 확인")
print("3. 듀오 시너지: 캐릭터 쌍의 공동 등장 성과와 표본 규모 확인")
print("4. 패치 타임라인: 버전별 메타 변화와 지속 상승/하락 흐름 확인")
print("5. ML 보정: 실제 성과와 기대 성과 차이로 과대/과소평가 구조 탐색")


## 10. 결론

분석 결과, 상위권 스쿼드 성과는 단일 캐릭터 티어보다 역할 조합, 듀오 시너지, 패치 변화가 함께 설명하는 구조로 나타난다.

대시보드는 이 결과를 정적인 보고서가 아니라 직접 탐색 가능한 형태로 제공하며, 사용자는 역할 조합을 먼저 고른 뒤 캐릭터와 듀오, 패치 흐름을 이어서 확인할 수 있다.

머신러닝 결과는 대시보드 내에서 기대 성과 대비 실제 성과가 높은 조합을 찾는 보조 지표로 활용 가능하다.
